In [1]:
from pyspark.sql.functions import current_timestamp,input_file_name, lit 
from datetime import datetime
# Batch ID for this notebook run
batch_id = datetime.now().strftime("%Y%m%d%H%M%S")

print(f"starting Bronze load. batch id :{batch_id}")


StatementMeta(, e68b2d07-72ca-47f0-b527-eb197f3c8c20, 3, Finished, Available, Finished, False)

starting Bronze load. batch id :20260718214034


In [2]:
# Creating schemas
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

print("Schemas created: bronze, silver, gold")


StatementMeta(, e68b2d07-72ca-47f0-b527-eb197f3c8c20, 4, Finished, Available, Finished, False)

Schemas created: bronze, silver, gold


In [3]:
# funtion to load csv files 
def load_csv_to_bronze(source_name, source_path, target_table, load_type):
    print(f"Loading CSV source: {source_name}")

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(source_path)
    )

    bronze_df = (
        df
        .withColumn("_source_name", lit(source_name))
        .withColumn("_source_file_name", input_file_name())
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_batch_id", lit(batch_id))
        .withColumn("_load_type", lit(load_type))
    )

    bronze_df.write.mode("overwrite").format("delta").option("overwriteSchema", "true")  .saveAsTable(target_table)

    print(f"Loaded {bronze_df.count()} rows into {target_table}")

StatementMeta(, e68b2d07-72ca-47f0-b527-eb197f3c8c20, 5, Finished, Available, Finished, False)

In [4]:
# creating function for json 
def load_json_to_bronze(source_name, source_path, target_table, load_type):
    print(f"Loading JSON source: {source_name}")

    df = (
        spark.read
        .format("json")
        .load(source_path)
    )

    bronze_df = (
        df
        .withColumn("_source_name", lit(source_name))
        .withColumn("_source_file_name", input_file_name())
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_batch_id", lit(batch_id))
        .withColumn("_load_type", lit(load_type))
    )

    bronze_df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(target_table)

    print(f"Loaded {bronze_df.count()} rows into {target_table}")

StatementMeta(, e68b2d07-72ca-47f0-b527-eb197f3c8c20, 6, Finished, Available, Finished, False)

In [5]:
# List of CSV sources to load into Bronze

csv_sources = [
    {
        "source_name": "orders",
        "source_path": "Files/retail_project/landing/orders/*.csv",
        "target_table": "bronze.orders",
        "load_type": "full_and_incremental_raw"
    },
    {
        "source_name": "customers",
        "source_path": "Files/retail_project/landing/customers/*.csv",
        "target_table": "bronze.customers",
        "load_type": "full_and_cdc_raw"
    },
    {
        "source_name": "products",
        "source_path": "Files/retail_project/landing/products/*.csv",
        "target_table": "bronze.products",
        "load_type": "full_and_cdc_raw"
    },
    {
        "source_name": "inventory",
        "source_path": "Files/retail_project/landing/inventory/*.csv",
        "target_table": "bronze.inventory",
        "load_type": "snapshot_raw"
    },
    {
        "source_name": "returns",
        "source_path": "Files/retail_project/landing/returns/*.csv",
        "target_table": "bronze.returns",
        "load_type": "incremental_raw"
    },
    {
        "source_name": "stores",
        "source_path": "Files/retail_project/landing/stores/*.csv",
        "target_table": "bronze.stores",
        "load_type": "full_raw"
    }
]

for source in csv_sources:
    load_csv_to_bronze(
        source_name=source["source_name"],
        source_path=source["source_path"],
        target_table=source["target_table"],
        load_type=source["load_type"]
    )

load_json_to_bronze(
    source_name="clickstream",
    source_path="Files/retail_project/landing/clickstream/*.json",
    target_table="bronze.clickstream",
    load_type="event_stream_raw"
)

StatementMeta(, e68b2d07-72ca-47f0-b527-eb197f3c8c20, 7, Finished, Available, Finished, False)

Loading CSV source: orders
Loaded 1213 rows into bronze.orders
Loading CSV source: customers
Loaded 129 rows into bronze.customers
Loading CSV source: products
Loaded 38 rows into bronze.products
Loading CSV source: inventory
Loaded 422 rows into bronze.inventory
Loading CSV source: returns
Loaded 81 rows into bronze.returns
Loading CSV source: stores
Loaded 8 rows into bronze.stores
Loading JSON source: clickstream
Loaded 753 rows into bronze.clickstream


In [6]:
display(spark.table("bronze.clickstream").limit(10))

StatementMeta(, e68b2d07-72ca-47f0-b527-eb197f3c8c20, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f2e5a064-f0cb-4452-a48e-dda4815ad103)